# 02: GO Analysis with R
**NOTE**: Run code in `01_preprocessing_with_python.ipynb` first to generate essential data files

1. Manual Annotation
2. Summary

---
## 0. Execution Environment Summary

In [ ]:
cat(
    system2(
        "bash",
        ".rep_exe_env_short.sh",
        stdout = TRUE,
        stderr = TRUE
    ),
    sep = "\n"
)

ENV_SUFFIX <- Sys.getenv("ENV_SUFFIX")

## 1. Manual Annotation
### 1-1. Enrichment Analysis

In [ ]:
suppressPackageStartupMessages({
    library(arrow)
    library(clusterProfiler)
    library(ComplexHeatmap)
    library(org.Hs.eg.db)
    library(tidyverse)
    devtools::load_all("basalcelldemo_rtools")
})

In [ ]:
degs <- read_parquet(glue::glue("data/degs_{ENV_SUFFIX}.pq")) %>%
  mutate(group = factor(group, levels = sort(unique(group))))
head(degs)

In [ ]:
comp_go <- compareCluster(
    geneClusters = names ~ group,
    data = degs,
    fun = "enrichGO",
    OrgD = org.Hs.eg.db,
    keyType = "SYMBOL",
    ont = "BP",
    pAdjustMethod = "BH",
    pvalueCutoff = 0.05,
    qvalueCutoff = 0.05
)

In [ ]:
figsize <- list(width = 8, height = 12)
options(repr.plot.width = figsize$width, repr.plot.height = figsize$height)

fig <- dotplot(comp_go, showCategory = 3) +
  theme_bw() +
  theme(
    axis.text.x = element_text(size = 10),
    axis.text.y = element_text(size = 10)
  ) +
  labs(title = glue::glue("BasalCellDemo ({ENV_SUFFIX})"), x = "Leiden", y = "Biological Processes")

print(fig)

ggsave(
    glue::glue("output/go_{ENV_SUFFIX}.pdf"), plot = fig, 
    width = figsize$width, height = figsize$height, units = "in")

### 1-2. Manual Annotation

In [ ]:
annotations <- read_parquet(glue::glue("data/annotations_{ENV_SUFFIX}.pq"))
head(annotations)

In [ ]:
manual_annot <- c(
    "0" = "Naïve/Memory T cells #1",
    "1" = "Naïve/Memory T cells #2",
    "2" = "B cells",
    "3" = "Cytotoxic T cells",
    "4" = "Monocytes #1",
    "5" = "Monocytes #2",
    "6" = "Undefined"
)

annotations <- annotations %>%
    mutate(
        manual_annotation = manual_annot[leiden]
    )
head(annotations)

---
## 2. Summary

In [ ]:
expressions <- read_parquet(glue::glue("data/expressions_{ENV_SUFFIX}.pq"))
head(expressions)

In [ ]:
top_degs <- read_parquet(glue::glue("data/top_degs_{ENV_SUFFIX}.pq"))
head(top_degs)

- creating sorted `setNames` with an original function `sorted_setnames` to configure legends in `ComplexHeatmap`

In [ ]:
auto_colors <- sorted_setnames(
    annotations$leiden_colors, 
    annotations$annotation, 
    annotations$leiden
)

celltypist_colors <- sorted_setnames(
    annotations$celltypist_colors, 
    annotations$celltypist_majority_voting, 
    annotations$leiden
)

manu_colors <- sorted_setnames(
    annotations$leiden_colors,
    annotations$manual_annotation,
    annotations$leiden
)

leiden_colors <- sorted_setnames(
    annotations$leiden_colors,
    annotations$leiden,
    annotations$leiden
)

In [ ]:
ha <- rowAnnotation(
    CellTypist = annotations$celltypist_majority_voting,
    Automated = annotations$annotation,
    Manual = annotations$manual_annotation,
    col = list(
        CellTypist = celltypist_colors,
        Automated = auto_colors,
        Manual = manu_colors
    ),
    annotation_legend_param = list(
        CellTypist = list(at = names(celltypist_colors)),
        Automated = list(at = names(auto_colors)),
        Manual = list(at = names(manu_colors))
    )
)

va <- columnAnnotation(
    LeidenDEGs = top_degs$group,
    col = list(
        LeidenDEGs = leiden_colors
    ),
    annotation_legend_param = list(
        LeidenDEGs = list(at = names(leiden_colors))
    )
)

mat <- as.matrix(expressions[, top_degs$names])
rownames(mat) <- expressions$index

barcode_sorted <- order(annotations$leiden, annotations$celltypist_majority_voting)

In [ ]:
figsize <- list(width = 12, height = 9)
options(repr.plot.width = figsize$width, repr.plot.height = figsize$height)

fig <- Heatmap(
    mat, top_annotation = va, right_annotation = ha, use_raster = TRUE,
    show_row_names = FALSE, row_order = barcode_sorted, cluster_columns = FALSE,
    name = "complexheatmap", heatmap_legend_param = list(title = "ln(RP10k+1)")
)

fig_title <- glue::glue("BasalCellDemo ({ENV_SUFFIX})")

pdf(glue::glue("output/complexheatmap_{ENV_SUFFIX}.pdf"), width = figsize$width, height = figsize$height)
draw(
    fig, column_title = fig_title, column_title_gp = grid::gpar(fontsize = 16, fontface = "bold"),
    padding = grid::unit(c(8, 2, 2, 2), "mm")
)
dev.off()

draw(
    fig, column_title = fig_title, column_title_gp = grid::gpar(fontsize = 16, fontface = "bold"),
    padding = grid::unit(c(8, 2, 2, 2), "mm")
)

---
## 3. Execution Environment Details

In [ ]:
cat(
    system2(
        "bash",
        ".rep_exe_env.sh",
        stdout = TRUE,
        stderr = TRUE
    ),
    sep = "\n"
)